In [9]:
import json
import pandas as pd

# Load data from JSON files
print("Loading IBIT data from JSON files...")
with open('output/ibit_data.json', 'r') as f:
    ibit_data = json.load(f)

with open('output/ibit_options.json', 'r') as f:
    ibit_options = json.load(f)

# Get current price and expirations
current_price = ibit_data['current_price']
expirations = ibit_data['expirations']

print(f"Current IBIT Price: ${current_price:.2f}")
print(f"Found {len(expirations)} expiration dates\n")

# Sort expirations and get the last and second to last
expirations_sorted = sorted(expirations)
last_expiration = expirations_sorted[-1]
second_last_expiration = expirations_sorted[-2]

print(f"Last expiration: {last_expiration}")
print(f"Second to last expiration: {second_last_expiration}\n")

#Load Sata Treasury Data
print("Loading Sata Treasury Data...")
with open('output/treasury_extracted_data.json', 'r') as f:
    treasury_extracted_data = json.load(f)

print(f"Sata Price: ${treasury_extracted_data['sata_price']:.2f}")

# Function to get puts below current price for a given expiration
def get_puts_below_price(expiration_date, current_price, options_data):
    """Get all puts with strike below current price for a given expiration"""
    if expiration_date not in options_data:
        return pd.DataFrame()
    
    puts_list = options_data[expiration_date].get('puts', [])
    
    if not puts_list:
        return pd.DataFrame()
    
    # Convert to DataFrame
    puts_df = pd.DataFrame(puts_list)
    
    # Filter puts where strike < current_price and strike is a multiple of 5
    puts_below = puts_df[(puts_df['strike'] < current_price) & (puts_df['strike'] % 5 == 0)].copy()
    
    # Calculate mid point price (bid + ask) / 2
    puts_below['mid_price'] = (puts_below['bid'] + puts_below['ask']) / 2
    
    # Select relevant columns (include model Greeks if ibit_option_deltas.py was run on ibit_data)
    base_cols = ['strike', 'bid', 'ask', 'mid_price', 'volume', 'openInterest', 'lastPrice']
    extra = [c for c in ('delta', 'gamma', 'rho', 'risk_free_rate', 'implied_volatility') if c in puts_below.columns]
    result = puts_below[base_cols + extra].copy()
    
    return result

# Option chains: prefer ibit_data.options_data (includes delta after ibit_option_deltas.py)
ibit_chain = ibit_data.get('options_data') or ibit_options

# Get puts for last expiration and second to last expiration
puts_last = get_puts_below_price(last_expiration, current_price, ibit_chain)
puts_second_last = get_puts_below_price(second_last_expiration, current_price, ibit_chain)

# Combine data and calculate theta
print("\n" + "="*70)
print("COMBINED TABLE WITH THETA ESTIMATION")
print("="*70)

# Merge the two dataframes on strike
if len(puts_last) > 0 and len(puts_second_last) > 0:
    # Create a merged dataframe (include last-expiry put delta when available)
    pl = puts_last[['strike', 'mid_price']].rename(columns={'mid_price': 'mid_price_last'})
    if 'delta' in puts_last.columns:
        pl['delta_last'] = puts_last['delta'].values
    merged = pd.merge(
        pl,
        puts_second_last[['strike', 'mid_price']].rename(columns={'mid_price': 'mid_price_second_last'}),
        on='strike',
        how='inner'
    )
    
    # Calculate difference in days between expiration dates
    from datetime import datetime
    last_exp_date = datetime.strptime(last_expiration, '%Y-%m-%d')
    second_last_exp_date = datetime.strptime(second_last_expiration, '%Y-%m-%d')
    days_difference = (last_exp_date - second_last_exp_date).days
    
    print(f"\nDays between expirations: {days_difference} days")
    print(f"Last expiration: {last_expiration}")
    print(f"Second to last expiration: {second_last_expiration}\n")
    
    # Calculate difference in mid price
    merged['mid_price_diff'] = merged['mid_price_last'] - merged['mid_price_second_last']
    
    # Calculate estimated theta (price decay per day)
    # Theta = (mid_price_last - mid_price_second_last) / days_difference
    merged['estimated_theta'] = merged['mid_price_diff'] / days_difference
    
    # Sort by strike
    merged = merged.sort_values('strike').reset_index(drop=True)
    
    # Create final table with requested columns
    out_cols = ['strike', 'mid_price_last', 'mid_price_second_last', 'mid_price_diff', 'estimated_theta']
    out_names = ['Strike', 'Mid Price (Last)', 'Mid Price (2nd to Last)', 'Difference in Mid Price', 'Estimated Theta ($/day)']
    if 'delta_last' in merged.columns:
        out_cols.append('delta_last')
        out_names.append('Put Delta (Last Exp)')
    result_table = merged[out_cols].copy()
    result_table.columns = out_names
    if 'Put Delta (Last Exp)' in result_table.columns:
        result_table['Put Delta (Last Exp)'] = result_table['Put Delta (Last Exp)'].round(4)
    
    print(result_table.to_string(index=False))
    
    print(f"\n" + "="*70)
    print("SUMMARY")
    print("="*70)
    print(f"\nCurrent IBIT Price: ${current_price:.2f}")
else:
    print("Cannot create combined table - missing data from one or both expirations.")

Loading IBIT data from JSON files...
Current IBIT Price: $36.06
Found 26 expiration dates

Last expiration: 2028-12-15
Second to last expiration: 2028-06-16

Loading Sata Treasury Data...
Sata Price: $98.34

COMBINED TABLE WITH THETA ESTIMATION

Days between expirations: 182 days
Last expiration: 2028-12-15
Second to last expiration: 2028-06-16

 Strike  Mid Price (Last)  Mid Price (2nd to Last)  Difference in Mid Price  Estimated Theta ($/day)
    5.0             0.305                    0.230                    0.075                 0.000412
   10.0             0.895                    0.640                    0.255                 0.001401
   15.0             1.840                    1.430                    0.410                 0.002253
   20.0             3.200                    2.590                    0.610                 0.003352
   25.0             4.900                    4.175                    0.725                 0.003984
   30.0             7.100                    6

In [10]:
# Constants
COST_OF_CAPITAL = 0.0464
MARGINAL_TAX_RATE = 0.25

# Calculate hedge amount
hedge_amount = treasury_extracted_data['sata_price'] - ((treasury_extracted_data['cash'] - treasury_extracted_data['debt'])*100/treasury_extracted_data['sata_notional'])

# Calculate contracts needed for hedging and annualized theta cost


# Use the mid price from the last expiration for calculations
if len(puts_last) > 0 and len(puts_second_last) > 0:
    # Create a dataframe with strike and mid_price from last expiration
    hedge_calc = puts_last[['strike', 'mid_price']].copy()
    
    # Calculate number of contracts needed
    # Formula: (strike - mid_price) * num_contracts = MAX_HEDGE_AMOUNT
    # So: num_contracts = MAX_HEDGE_AMOUNT / (strike - mid_price)
    hedge_calc['contracts_needed'] = hedge_amount / (hedge_calc['strike'] - hedge_calc['mid_price'])
    
    # Merge with theta data
    hedge_calc = pd.merge(
        hedge_calc,
        merged[['strike', 'estimated_theta']],
        on='strike',
        how='inner'
    )

    gcols = [c for c in ('delta', 'gamma', 'rho', 'risk_free_rate', 'implied_volatility') if c in puts_last.columns]
    if gcols:
        hedge_calc = pd.merge(hedge_calc, puts_last[['strike'] + gcols], on='strike', how='left')
    
    # Calculate annualized theta cost
    # Annualized theta = estimated_theta * contracts_needed * 365.25
    hedge_calc['annualized_theta_cost'] = hedge_calc['estimated_theta'] * hedge_calc['contracts_needed'] * 365.25
    
    # Calculate annualized cost of capital
    # Annualized cost of capital = contracts_needed * mid_price * COST_OF_CAPITAL
    hedge_calc['cost_of_hedge_options'] = hedge_calc['contracts_needed'] * hedge_calc['mid_price']
    hedge_calc['cost_of_hedged_share'] = hedge_calc['cost_of_hedge_options'] + treasury_extracted_data['sata_price']
    hedge_calc['annualized_cost_of_capital'] = hedge_calc['cost_of_hedged_share'] * COST_OF_CAPITAL
    
    # Calculate annualized cost as percentage of max hedge amount
    # Annualized cost = annualized_cost_of_capital + annualized_theta_cost
    hedge_calc['annualized_cost_pct'] = (hedge_calc['annualized_cost_of_capital'] + hedge_calc['annualized_theta_cost'])

    # Calculate annualized cost after tax
    hedge_calc['annualized_cost_after_tax_pct'] = hedge_calc['annualized_cost_pct'] * (1 - MARGINAL_TAX_RATE)
    # Sort by strike
    hedge_calc = hedge_calc.sort_values('strike').reset_index(drop=True)
    
    # Create display table
    display_table = hedge_calc[['strike', 'mid_price', 'contracts_needed', 'annualized_cost_of_capital', 'annualized_theta_cost', 'annualized_cost_pct', 'annualized_cost_after_tax_pct']].copy()
    display_table.columns = ['Strike', 'Mid Price', 'Contracts Needed', 'Annualized COC', 'Annualized Theta', 'Annualized Cost (%)', 'Annualized Cost After Tax (%)']
    
    print("="*70)
    print("HEDGING ANALYSIS - CONTRACTS NEEDED & ANNUALIZED THETA COST")
    print("="*70)
    print(f"Cost of Capital: {COST_OF_CAPITAL * 100}%")
    print(f"Marginal Tax Rate: {MARGINAL_TAX_RATE * 100}%")
    print(f"Hedge Amount: ${hedge_amount:.2f}\n")
    print(display_table.to_string(index=False))

    hedge_calc

else:
    print("Cannot calculate hedging - missing data.")


HEDGING ANALYSIS - CONTRACTS NEEDED & ANNUALIZED THETA COST
Cost of Capital: 4.64%
Marginal Tax Rate: 25.0%
Hedge Amount: $73.53

 Strike  Mid Price  Contracts Needed  Annualized COC  Annualized Theta  Annualized Cost (%)  Annualized Cost After Tax (%)
    5.0      0.305         15.662253        4.784702          2.357406             7.142108                       5.356581
   10.0      0.895          8.076252        4.898441          4.133033             9.031474                       6.773605
   15.0      1.840          5.587711        5.040107          4.597658             9.637764                       7.228323
   20.0      3.200          4.377040        5.212953          5.358327            10.571280                       7.928460
   25.0      4.900          3.658422        5.394829          5.322928            10.717757                       8.038318
   30.0      7.100          3.211104        5.620916          5.799836            11.420752                       8.565564
   35.0  

In [11]:
# Constants — $10,000 book on SATA + listed puts (last expiry)
CAPITAL = 10_000
SATA_MARGIN_REQUIREMENT = 0.15

optimal_strike = (ibit_data['current_price'] * (treasury_extracted_data['debt'] + treasury_extracted_data['sata_notional']) / (treasury_extracted_data['btc_holdings'] * ibit_data['current_price'] / ibit_data['btc_per_share'] + treasury_extracted_data['cash']))

# STRIKE_TICK = 5
# optimal_strike_floor   = math.floor(optimal_strike / STRIKE_TICK) * STRIKE_TICK
# optimal_strike_ceil    = math.ceil(optimal_strike  / STRIKE_TICK) * STRIKE_TICK
# optimal_strike_nearest = round(optimal_strike      / STRIKE_TICK) * STRIKE_TICK

print(f"Initial Capital:              ${CAPITAL:,.2f}")
print(f"SATA Margin Requirement:      {SATA_MARGIN_REQUIREMENT * 100:.0f}%")
print(f"Optimal Strike (continuous):  ${optimal_strike:.2f}")

hedge_calc['capital_per_share'] = (
    hedge_calc['mid_price'] * hedge_calc['contracts_needed']
    + SATA_MARGIN_REQUIREMENT * treasury_extracted_data['sata_price']
)
hedge_calc['num_shares'] = CAPITAL / hedge_calc['capital_per_share']
hedge_calc['dividend'] = hedge_calc['num_shares'] * treasury_extracted_data['sata_effective_yield']
hedge_calc['total_purchase'] = hedge_calc['num_shares'] * (treasury_extracted_data['sata_price'] + hedge_calc['cost_of_hedge_options'])
hedge_calc['borrow_cost'] = -(hedge_calc['total_purchase'] - CAPITAL) * COST_OF_CAPITAL
hedge_calc['theta_cost'] = -hedge_calc['annualized_theta_cost'] * hedge_calc['num_shares']
hedge_calc['tax_benefit'] = (-MARGINAL_TAX_RATE) * (hedge_calc['theta_cost'] + hedge_calc['borrow_cost'])
hedge_calc['total_yield'] = hedge_calc['dividend'] + hedge_calc['theta_cost'] + hedge_calc['borrow_cost'] + hedge_calc['tax_benefit']
hedge_calc['total_cash_flow'] = hedge_calc['dividend'] + hedge_calc['borrow_cost'] + hedge_calc['theta_cost']

display_table = hedge_calc[['strike', 'num_shares', 'total_purchase', 'dividend', 'borrow_cost', 'theta_cost', 'tax_benefit', 'total_yield', 'total_cash_flow']].copy()
display_table['num_shares'] = display_table['num_shares'].round(2)
display_table['dividend'] = display_table['dividend'].round(2)
display_table['total_purchase'] = display_table['total_purchase'].round(2)
display_table['borrow_cost'] = display_table['borrow_cost'].round(2)
display_table['theta_cost'] = display_table['theta_cost'].round(2)
display_table['tax_benefit'] = display_table['tax_benefit'].round(2)
display_table['total_yield'] = display_table['total_yield'].round(2)
display_table['total_cash_flow'] = display_table['total_cash_flow'].round(2)
display_table.columns = ['Strike', 'Num shares', 'Total Purchase', 'Dividend', 'Borrow Cost', 'Theta Cost', 'Tax Benefit', 'Total Yield', 'Total Cash Flow']

print(display_table.to_string(index=False))
print()

# --- Option book P&L (last table): same $10k book ---
# Listed option contracts = (num_shares × contracts_needed) / 100 — each contract is 100 shares of IBIT.
# P&L = N_listed × 100 × (V(S₁,r₁) − V(S₀,r₀)) with S₁=S₀(1+h), r₁=r(1+h), same σ=IV (full CRR; avoids Taylor blow-up at ±50%).
import math
from datetime import date, datetime, timezone

from fetch_treasury_zero_yieldcurve import TreasuryZeroCurve
from ibit_option_deltas import (
    DEFAULT_DIV_YIELD,
    DEFAULT_TREE_STEPS,
    american_price_crr,
    _parse_valuation_datetime,
    year_fraction_to_expiry,
)


def _load_treasury_zero_curve_notebook():
    """Same curve as pricing: embedded in ibit_data, else yield_curve.json from fetch_data."""
    raw = ibit_data.get("treasury_zero_curve")
    if isinstance(raw, dict) and raw.get("pillars_years"):
        try:
            return TreasuryZeroCurve.from_saved_dict(raw), "ibit_data['treasury_zero_curve']"
        except Exception:
            pass
    try:
        with open("output/yield_curve.json") as _f:
            raw = json.load(_f)
        if isinstance(raw, dict) and raw.get("pillars_years"):
            return TreasuryZeroCurve.from_saved_dict(raw), "yield_curve.json"
    except Exception:
        pass
    return None, None


# Shock h ∈ {-50%, …, +50%} in 10% steps (parallel): S₁=S₀(1+h), r₁=r(1+h); σ fixed at row implied vol.
# N_listed = (num_shares × contracts_needed) / 100 option contracts; P&L = N_listed·100·ΔV (dollars).
# r is Treasury r_eq(T) (continuous), never COST_OF_CAPITAL (borrowing).
CONTRACT_MULT = 100
SHOCK_PCTS = list(range(-50, 51, 10))
S0 = float(current_price)

if 'hedge_calc' in dir() and len(hedge_calc) > 0 and 'contracts_needed' in hedge_calc.columns and 'num_shares' in hedge_calc.columns:
    if 'delta' in hedge_calc.columns and hedge_calc['delta'].notna().any():
        curve_rf, curve_src = _load_treasury_zero_curve_notebook()
        ts_ibit = ibit_data.get("timestamp")
        valuation_dt = (
            _parse_valuation_datetime(str(ts_ibit))
            if ts_ibit
            else datetime.now(timezone.utc)
        )
        T_last = None
        r_eq_last = None
        if "last_expiration" in dir():
            try:
                T_last = year_fraction_to_expiry(
                    valuation_dt, date.fromisoformat(last_expiration)
                )
            except Exception:
                T_last = None
        if curve_rf is not None and T_last is not None:
            r_eq_last = curve_rf.equivalent_constant_rate(T_last)
            if not math.isfinite(r_eq_last):
                r_eq_last = None

        rows = []
        if 'implied_volatility' not in hedge_calc.columns or not hedge_calc['implied_volatility'].notna().any():
            print(
                'WARNING: implied_volatility missing on hedge_calc — P&L uses Taylor Δ+½Γ+ρ '
                '(unreliable for large ±% shocks). Re-run cell 0 so puts_last carries IV, then cells 1–2.'
            )
        for _, row in hedge_calc.iterrows():
            n_sh = float(row['num_shares'])
            n_per_share = float(row['contracts_needed'])
            n_listed = n_sh * n_per_share / CONTRACT_MULT
            rec = {
                'Strike': row['strike'],
                'Delta': float('nan'),
                'Num shares': round(n_sh, 2),
                'Option contracts': round(n_listed, 4),
            }
            if pd.isna(row.get('delta')):
                for p in SHOCK_PCTS:
                    rec[f'{p:+d}%'] = float('nan')
                rows.append(rec)
                continue
            dlt = float(row['delta'])
            rec['Delta'] = round(dlt, 4)
            if pd.notna(row.get('risk_free_rate')):
                ri = float(row['risk_free_rate'])
            elif r_eq_last is not None:
                ri = r_eq_last
            else:
                ri = float('nan')
            iv_raw = row.get('implied_volatility')
            use_full = (
                T_last is not None
                and pd.notna(iv_raw)
                and float(iv_raw) > 0
                and math.isfinite(ri)
            )
            if use_full:
                K = float(row['strike'])
                sig = float(iv_raw)
                V0 = american_price_crr(
                    S0, K, T_last, ri, DEFAULT_DIV_YIELD, sig,
                    n_steps=DEFAULT_TREE_STEPS, is_call=False,
                )
                if not math.isfinite(V0):
                    for p in SHOCK_PCTS:
                        rec[f'{p:+d}%'] = float('nan')
                    rows.append(rec)
                    continue
                for p in SHOCK_PCTS:
                    h = p / 100.0
                    S_new = S0 * (1.0 + h)
                    r_new = ri * (1.0 + h)
                    V1 = american_price_crr(
                        S_new, K, T_last, r_new, DEFAULT_DIV_YIELD, sig,
                        n_steps=DEFAULT_TREE_STEPS, is_call=False,
                    )
                    rec[f'{p:+d}%'] = (
                        n_listed * CONTRACT_MULT * (V1 - V0)
                        if math.isfinite(V1) else float('nan')
                    )
            else:
                gam = float(row['gamma']) if pd.notna(row.get('gamma')) else 0.0
                rho = float(row['rho']) if pd.notna(row.get('rho')) else 0.0
                for p in SHOCK_PCTS:
                    h = p / 100.0
                    dS = h * S0
                    dr = h * ri
                    dPi = dlt * dS + 0.5 * gam * dS * dS + rho * dr
                    rec[f'{p:+d}%'] = n_listed * CONTRACT_MULT * dPi
            rows.append(rec)
        sens_df = pd.DataFrame(rows)
        # Maintainer note: local Δ+½Γ+ρ Taylor at ±50% spot can flip sign for long puts (½ΓdS² dominates).
        # When implied_volatility is present we use full CRR repricing at (S,r) with fixed σ.
        print('=' * 70)
        print('OPTION BOOK P&L ($) — parallel shock -50% … +50% (10% steps), $10k capital book')
        print('Option contracts = (Num shares × contracts_needed) / 100 (100 shares per contract).')
        print('Each shock: full CRR reprice at S₁=S₀(1+h), r₁=r(1+h), σ=row IV (fallback: Δ+½Γ+ρ if no IV).')
        if curve_src and r_eq_last is not None and T_last is not None:
            exp_lbl = last_expiration if "last_expiration" in dir() else "?"
            print(
                f'S₀ = ${S0:.2f}  |  r for ρ: row risk_free_rate (Treasury from pricing) '
                f'else r_eq(T={exp_lbl}, T={T_last:.4f}y)={r_eq_last:.4f} from {curve_src}'
            )
        elif curve_src:
            print(
                f'S₀ = ${S0:.2f}  |  r for ρ: row risk_free_rate only '
                f'(could not align curve to last_expiration)'
            )
        else:
            print(
                f'S₀ = ${S0:.2f}  |  r for ρ: row risk_free_rate only — '
                f'run fetch_data.py / ibit_option_deltas.py so ibit_data or yield_curve.json has the Treasury curve'
            )
        print('=' * 70)
        _sens_disp = sens_df.round(2)
        _sens_disp['Delta'] = sens_df['Delta'].round(4)
        print(_sens_disp.to_string(index=False))
        print()
    else:
        print('P&L sensitivity skipped: no delta on hedge_calc — run ibit_option_deltas.py, then re-run cells 0–1.\n')
else:
    print('P&L sensitivity skipped: hedge_calc missing or empty — run cells 0–1 first, then this cell.\n')

Initial Capital:              $10,000.00
SATA Margin Requirement:      15%
Optimal Strike (continuous):  $19.40
 Strike  Num shares  Total Purchase  Dividend  Borrow Cost  Theta Cost  Tax Benefit  Total Yield  Total Cash Flow
    5.0      512.08        52804.89   6769.29     -1986.15    -1207.18       798.33      4374.30          3575.97
   10.0      454.97        48031.08   6014.35     -1764.64    -1880.40       911.26      3280.56          2369.30
   15.0      399.48        43392.56   5280.80     -1549.41    -1836.67       846.52      2741.24          1894.72
   20.0      347.73        39067.05   4596.75     -1348.71    -1863.26       802.99      2187.77          1384.78
   25.0      306.02        35580.40   4045.36     -1186.93    -1628.93       703.96      1933.47          1229.50
   30.0      266.31        32261.04   3520.43     -1032.91    -1544.56       644.37      1587.32           942.95
   35.0      235.60        29694.23   3114.50      -913.81    -1264.95       544.69      1